# Regeneración mensual del forecast de demanda

Sobreescribe `workspace.gasifera.ml_forecast_demanda` con la proyección a 6 meses (regresión lag-12 + estacionalidad sin/cos + tendencia), TOTAL + 8 regiones, entrenada con todo el historial disponible de `gold_consumo_mensual`.

Lo ejecuta el job programado **Gasifera Regenerar Forecast** (día 2 de cada mes, 05:00 America/Bogota), un día antes de la alerta de desviación (día 3), para que dashboard y alerta lean siempre el forecast fresco.

In [0]:
# Regenera la proyección de demanda a 6 meses (TOTAL + regiones) y sobreescribe la tabla
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from pyspark.sql import functions as F

FEATS = ["lag_12", "sin_mes", "cos_mes", "tendencia"]
HORIZONTE = 6

serie = spark.sql("""
SELECT periodo, SUM(volumen_m3) AS volumen_m3
FROM workspace.gasifera_gold.gold_consumo_mensual
GROUP BY periodo ORDER BY periodo""").toPandas()
serie["periodo"] = pd.to_datetime(serie["periodo"])
serie = serie.sort_values("periodo").reset_index(drop=True)
serie["lag_12"] = serie.volumen_m3.shift(12)
serie["sin_mes"] = np.sin(2 * np.pi * serie.periodo.dt.month / 12)
serie["cos_mes"] = np.cos(2 * np.pi * serie.periodo.dt.month / 12)
serie["tendencia"] = np.arange(len(serie))

future_dates = pd.date_range(serie.periodo.max() + pd.offsets.MonthBegin(1), periods=HORIZONTE, freq="MS")
last_tend = serie.tendencia.iloc[-1]
filas = []

# Total
sc = serie.dropna(subset=FEATS)
mf = LinearRegression().fit(sc[FEATS], sc.volumen_m3)
for i, p in enumerate(future_dates, start=1):
    lag12 = float(serie.loc[serie.periodo == p - pd.DateOffset(months=12), "volumen_m3"].iloc[0])
    x = pd.DataFrame([{"lag_12": lag12, "sin_mes": np.sin(2 * np.pi * p.month / 12), "cos_mes": np.cos(2 * np.pi * p.month / 12), "tendencia": last_tend + i}])
    filas.append({"periodo": p, "region": "TOTAL", "prediccion_m3": float(mf.predict(x[FEATS])[0]), "referencia_12m_m3": lag12})

# Por región
reg = spark.sql("""
SELECT periodo, region, SUM(volumen_m3) AS volumen_m3
FROM workspace.gasifera_gold.gold_consumo_mensual GROUP BY periodo, region""").toPandas()
reg["periodo"] = pd.to_datetime(reg["periodo"])
for r, g in reg.groupby("region"):
    g = g.sort_values("periodo").reset_index(drop=True)
    g["lag_12"] = g.volumen_m3.shift(12)
    g["sin_mes"] = np.sin(2 * np.pi * g.periodo.dt.month / 12)
    g["cos_mes"] = np.cos(2 * np.pi * g.periodo.dt.month / 12)
    g["tendencia"] = np.arange(len(g))
    tr = g.dropna(subset=FEATS)
    if len(tr) < 12:
        print(f"Region {r} con menos de 12 meses limpios: excluida")
        continue
    mr = LinearRegression().fit(tr[FEATS], tr.volumen_m3)
    for i, p in enumerate(future_dates, start=1):
        lag12 = float(g.loc[g.periodo == p - pd.DateOffset(months=12), "volumen_m3"].iloc[0])
        x = pd.DataFrame([{"lag_12": lag12, "sin_mes": np.sin(2 * np.pi * p.month / 12), "cos_mes": np.cos(2 * np.pi * p.month / 12), "tendencia": g.tendencia.iloc[-1] + i}])
        filas.append({"periodo": p, "region": r, "prediccion_m3": float(mr.predict(x[FEATS])[0]), "referencia_12m_m3": lag12})

fc = pd.DataFrame(filas)
fc["modelo"] = "regresion_lag12_estacionalidad_tendencia"
fc["generado"] = str(pd.Timestamp.utcnow().date())

sdf = spark.createDataFrame(fc).withColumn("periodo", F.to_date("periodo"))
sdf.write.mode("overwrite").saveAsTable("workspace.gasifera.ml_forecast_demanda")

chk = spark.sql("SELECT region, COUNT(*) n, ROUND(SUM(prediccion_m3)/1e6, 1) total_Mm3 FROM workspace.gasifera.ml_forecast_demanda GROUP BY region ORDER BY total_Mm3 DESC")
display(chk)
print(f"Regenerado: {future_dates[0].date()} -> {future_dates[-1].date()} | {len(fc)} filas")